# Solution 4D: Time-Series Cross-Validation
**BUSI 722: Data-Driven Finance II**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from scipy.stats import spearmanr

df = pd.read_parquet("merged.parquet")
df = df[df["month"] >= "2023-01"].copy()

FEATURES = ["momentum", "lag_month", "pb", "roe", "grossmargin",
            "assetturnover", "leverage", "asset_growth", "gp_to_assets"]

for feat in FEATURES:
    df[f"{feat}_rank"] = df.groupby("month")[feat].transform(lambda x: x.rank(pct=True))
df["ret_rank"] = df.groupby("month")["return"].transform(lambda x: x.rank(pct=True))
feat_cols = [f"{f}_rank" for f in FEATURES]
months = sorted(df["month"].unique())
print(f"Data: {len(df):,} rows, {len(months)} months ({months[0]} to {months[-1]})")

Data: 103,931 rows, 35 months (2023-01 to 2025-11)


## 1. 4-fold time-series CV within Jan 2023 - Dec 2024

In [2]:
cv_folds = [
    ("2023-01", "2023-06", "2023-07", "2023-12"),
    ("2023-01", "2023-12", "2024-01", "2024-06"),
    ("2023-01", "2024-06", "2024-07", "2024-12"),
    ("2023-07", "2024-06", "2024-07", "2024-12"),
]

depths = [3, 5, 7]
cv_results = {d: [] for d in depths}

for fold_i, (ts, te, vs, ve) in enumerate(cv_folds):
    train_fold = df[(df["month"] >= ts) & (df["month"] <= te)]
    val_fold = df[(df["month"] >= vs) & (df["month"] <= ve)]
    if len(train_fold) < 100 or len(val_fold) < 100:
        continue
    for depth in depths:
        model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=depth,
                                  random_state=42, verbosity=-1, n_jobs=-1)
        model.fit(train_fold[feat_cols].values, train_fold["ret_rank"].values)
        preds = model.predict(val_fold[feat_cols].values)
        rho, _ = spearmanr(preds, val_fold["return"].values)
        cv_results[depth].append(rho)
    print(f"  Fold {fold_i+1}: Train {ts} to {te}, Validate {vs} to {ve}")

  Fold 1: Train 2023-01 to 2023-06, Validate 2023-07 to 2023-12


  Fold 2: Train 2023-01 to 2023-12, Validate 2024-01 to 2024-06


  Fold 3: Train 2023-01 to 2024-06, Validate 2024-07 to 2024-12


  Fold 4: Train 2023-07 to 2024-06, Validate 2024-07 to 2024-12


## 2. Results

In [3]:
print(f"{'Depth':>8s} {'Mean rho':>10s} {'Std rho':>10s}")
print("-" * 30)
for depth in depths:
    if cv_results[depth]:
        print(f"  {depth:5d}   {np.mean(cv_results[depth]):10.4f} {np.std(cv_results[depth]):10.4f}")

best = max(depths, key=lambda d: np.mean(cv_results[d]) if cv_results[d] else -1)
print(f"\nBest depth by CV: {best}")

   Depth   Mean rho    Std rho
------------------------------
      3       0.1127     0.0411
      5       0.1120     0.0410
      7       0.1130     0.0434

Best depth by CV: 7


## 3. Discussion

The improvement from cross-validation is typically small for LightGBM depth selection, as the model is relatively robust across depths 3-7. However, the procedure is important for more sensitive hyperparameters (e.g., learning rate, number of estimators) and ensures we are not overfitting to the test period.